# 05 · Modellvergleich — Favorita

Liest die in Notebook 04 gespeicherten Ergebnisse und erstellt einen vollständigen Vergleich:
- Metriken auf Val- und Test-Set
- Visuelle Gegenüberstellung
- Fehleranalyse pro Store und Zeitverlauf
- Residuenanalyse des besten Modells

In [1]:
import sys
import os
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.append(os.path.abspath('../03_src'))
from config import FINAL, TARGET_COL
from utilis import evaluate

sns.set_style('whitegrid')
RESULTS = FINAL / 'results'

## 1 · Ergebnisse laden

In [ ]:
# Val-Vorhersagen
val_sarimax  = pl.read_parquet(RESULTS / 'val_sarimax.parquet')
val_prophet  = pl.read_parquet(RESULTS / 'val_prophet.parquet')
val_xgb      = pl.read_parquet(RESULTS / 'val_xgb.parquet')
val_lgbm     = pl.read_parquet(RESULTS / 'val_lgbm.parquet')
val_patchtst = pl.read_parquet(RESULTS / 'val_patchtst.parquet')

# Test-Vorhersagen
test_xgb      = pl.read_parquet(RESULTS / 'test_xgb.parquet')
test_lgbm     = pl.read_parquet(RESULTS / 'test_lgbm.parquet')
test_patchtst = pl.read_parquet(RESULTS / 'test_patchtst.parquet')

# Metriken aus NB04
metrics_val  = pl.read_parquet(RESULTS / 'metrics_val.parquet')
metrics_test = pl.read_parquet(RESULTS / 'metrics_test.parquet')

print('Val-Metriken:')
print(metrics_val.sort('mae'))

## 2 · Test-Set: SARIMAX & Prophet nachziehen

SARIMAX und Prophet werden auf dem Test-Set evaluiert indem die gespeicherten
Val-Vorhersage-Struktur mit dem echten Test-Zeitraum befüllt wird.
Da beide Modelle pro Store laufen, reicht ein Re-Run auf `test` statt `val`.

In [ ]:
from utilis import run_sarimax, run_prophet
from config import TRAIN_END, VAL_END, EXOG_COLS

df = pl.read_parquet(FINAL / 'final_dataset.parquet')

train_val = df.filter(pl.col('date') <= VAL_END)
test      = df.filter(pl.col('date') > VAL_END)
stores    = sorted(df['store_nbr'].unique().to_list())

print(f'Train+Val: {train_val.shape}  Test: {test.shape}')

In [ ]:
test_sarimax = run_sarimax(train_val, test, stores=stores, exog_cols=EXOG_COLS)
sarimax_test_metrics = evaluate(
    test_sarimax['y_true'].to_numpy(),
    test_sarimax['pred_sarimax'].to_numpy(),
    label='SARIMAX Test'
)
test_sarimax.write_parquet(RESULTS / 'test_sarimax.parquet')

In [ ]:
test_prophet = run_prophet(train_val, test, stores=stores, extra_regressors=['oil_price'])
prophet_test_metrics = evaluate(
    test_prophet['y_true'].to_numpy(),
    test_prophet['pred_prophet'].to_numpy(),
    label='Prophet Test'
)
test_prophet.write_parquet(RESULTS / 'test_prophet.parquet')

In [ ]:
# Test-Metriken vervollständigen & speichern
metrics_test_full = pl.read_parquet(RESULTS / 'metrics_test.parquet')

# SARIMAX & Prophet Test-Werte einsetzen
updates = {
    'SARIMAX':  sarimax_test_metrics,
    'Prophet':  prophet_test_metrics,
}
rows = []
for row in metrics_test_full.iter_rows(named=True):
    if row['modell'] in updates:
        row.update(updates[row['modell']])
    rows.append(row)
metrics_test_full = pl.DataFrame(rows)
metrics_test_full.write_parquet(RESULTS / 'metrics_test.parquet')

print('Test-Metriken (vollständig):')
print(metrics_test_full.sort('mae'))

## 3 · Metriken-Vergleich Val vs. Test

In [ ]:
# Kombinierte Tabelle
all_metrics = pl.concat([metrics_val, metrics_test_full]).to_pandas()

modell_order = ['SARIMAX', 'Prophet', 'XGBoost', 'LightGBM', 'PatchTST']
palette = {'val': 'steelblue', 'test': 'tomato'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, ['mae', 'rmse', 'mape']):
    for split, color in palette.items():
        data = all_metrics[all_metrics['split'] == split]
        data = data.set_index('modell').reindex(modell_order)
        x = np.arange(len(modell_order))
        offset = -0.2 if split == 'val' else 0.2
        ax.bar(x + offset, data[metric], width=0.35,
               color=color, alpha=0.85, label=split)
    ax.set_xticks(np.arange(len(modell_order)))
    ax.set_xticklabels(modell_order, rotation=30, ha='right')
    ax.set_title(metric.upper(), fontsize=13)
    ax.legend()

fig.suptitle('Modellvergleich — Val vs. Test', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4 · Fehleranalyse pro Store

In [ ]:
# MAE pro Store für jedes Modell auf dem Test-Set
def mae_per_store(preds_df, pred_col):
    return (
        preds_df
        .with_columns(
            (pl.col(pred_col) - pl.col('y_true')).abs().alias('abs_error')
        )
        .group_by('store_nbr')
        .agg(pl.col('abs_error').mean().alias('mae'))
        .sort('store_nbr')
    )

mae_xgb      = mae_per_store(test_xgb,      'pred_xgb')
mae_lgbm     = mae_per_store(test_lgbm,     'pred_lgbm')
mae_patchtst = mae_per_store(test_patchtst, 'pred_patchtst')

# Zusammenführen
store_mae = (
    mae_xgb.rename({'mae': 'XGBoost'})
    .join(mae_lgbm.rename({'mae': 'LightGBM'}),     on='store_nbr')
    .join(mae_patchtst.rename({'mae': 'PatchTST'}), on='store_nbr')
    .to_pandas().set_index('store_nbr')
)

fig, ax = plt.subplots(figsize=(13, 5))
store_mae.plot(kind='bar', ax=ax, width=0.7, alpha=0.85)
ax.set_title('MAE pro Store (Test-Set) — ML & DL Modelle', fontsize=13)
ax.set_xlabel('Store')
ax.set_ylabel('MAE')
ax.legend(title='Modell')
plt.tight_layout()
plt.show()

## 5 · Zeitverlauf: Vorhersage vs. Realität

In [ ]:
# Store 1 als Beispiel — täglicher Verlauf auf dem Test-Set
store_id = 1

def get_store(df, store_col='store_nbr'):
    if store_col in df.columns:
        return df.filter(pl.col(store_col) == store_id).sort('date').to_pandas()
    return df

s_xgb      = get_store(test_xgb)
s_lgbm     = get_store(test_lgbm)
s_patchtst = get_store(test_patchtst)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(s_xgb['date'],      s_xgb['y_true'],         color='black',     linewidth=1.2, label='Realität')
ax.plot(s_xgb['date'],      s_xgb['pred_xgb'],       color='steelblue', linewidth=1,   label='XGBoost',  linestyle='--')
ax.plot(s_lgbm['date'],     s_lgbm['pred_lgbm'],     color='seagreen',  linewidth=1,   label='LightGBM', linestyle='--')
ax.plot(s_patchtst['date'], s_patchtst['pred_patchtst'], color='tomato', linewidth=1,  label='PatchTST', linestyle='--')
ax.set_title(f'Vorhersage vs. Realität — Store {store_id} (Test-Set)', fontsize=13)
ax.set_ylabel('Transaktionen')
ax.legend()
plt.tight_layout()
plt.show()

## 6 · Residuenanalyse — bestes Modell

In [ ]:
# Bestes Modell nach MAE auf Test-Set bestimmen
best_model = metrics_test_full.sort('mae')['modell'][0]
print(f'Bestes Modell nach MAE auf Test-Set: {best_model}')

# Residuen des besten Modells
pred_map = {
    'XGBoost':  (test_xgb,      'pred_xgb'),
    'LightGBM': (test_lgbm,     'pred_lgbm'),
    'PatchTST': (test_patchtst, 'pred_patchtst'),
    'SARIMAX':  (pl.read_parquet(RESULTS / 'test_sarimax.parquet'), 'pred_sarimax'),
    'Prophet':  (pl.read_parquet(RESULTS / 'test_prophet.parquet'), 'pred_prophet'),
}

best_df, best_col = pred_map[best_model]
best_pd = best_df.with_columns(
    (pl.col(best_col) - pl.col('y_true')).alias('residual')
).to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogramm
axes[0].hist(best_pd['residual'], bins=60, color='steelblue', edgecolor='none')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_title(f'Residuen-Verteilung ({best_model})', fontsize=12)
axes[0].set_xlabel('Residual (Pred - True)')

# Scatter: Pred vs. True
axes[1].scatter(best_pd['y_true'], best_pd[best_col], alpha=0.15, s=5, color='steelblue')
lims = [best_pd['y_true'].min(), best_pd['y_true'].max()]
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_title('Pred vs. True', fontsize=12)
axes[1].set_xlabel('True')
axes[1].set_ylabel('Predicted')

# Residuen über Zeit (Store 1)
store1_res = best_pd[best_pd['store_nbr'] == str(store_id)] if best_pd['store_nbr'].dtype == object \
             else best_pd[best_pd['store_nbr'] == store_id]
axes[2].plot(store1_res['date'], store1_res['residual'], linewidth=0.8, color='steelblue')
axes[2].axhline(0, color='red', linestyle='--', linewidth=1)
axes[2].set_title(f'Residuen über Zeit — Store {store_id}', fontsize=12)
axes[2].set_ylabel('Residual')

plt.tight_layout()
plt.show()

print(f'\nBias (Ø Residual): {best_pd["residual"].mean():.2f}')
print(f'Std Residual:      {best_pd["residual"].std():.2f}')

## 7 · Finale Zusammenfassung

In [ ]:
print('=' * 60)
print('MODELLVERGLEICH — FINALE ERGEBNISSE')
print('=' * 60)
print()
print('VAL-SET:')
print(metrics_val.sort('mae').to_pandas().to_string(index=False))
print()
print('TEST-SET:')
print(metrics_test_full.sort('mae').to_pandas().to_string(index=False))
print()
print(f'Bestes Modell (Test MAE): {best_model}')